# Gemini に決まった形式（JSON スキーマ）で出力させる

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/13_llm_api/gemini_structured.ipynb)

NLP クックブック 13-04 ／ LLM の API（Gemini と Claude）

> API キーが必要（Colab 左の 🔑 シークレットに登録）

LLM の出力を、決まった項目を持つ JSON として確実に受け取りたい。テキストから情報を抜き出して、そのままプログラムで使いたい。

In [ ]:
!pip install -q google-genai pydantic

In [ ]:
from typing import Literal
from google import genai
from google.colab import userdata
from google.genai import types
from pydantic import BaseModel

client = genai.Client(api_key=userdata.get("GOOGLE_API_KEY"))
MODEL = "gemini-flash-latest"

class Entity(BaseModel):
    text: str
    label: Literal["人名", "組織名", "地名", "日付"]

class Extraction(BaseModel):
    entities: list[Entity]
    summary: str

text = "2024 年 4 月、山田花子氏は東京大学を離れ、京都にある理化学研究所の新しいチームに加わった。"
response = client.models.generate_content(
    model=MODEL,
    contents=f"次の文から固有表現を抽出し、1 文で要約してください。\n\n{text}",
    config=types.GenerateContentConfig(response_mime_type="application/json", response_schema=Extraction),
)
result: Extraction = response.parsed          # スキーマに沿って検証済みの Pydantic オブジェクト
for e in result.entities:
    print(e.label, e.text, "(原文にある)" if e.text in text else "(原文にない!)")
print(result.summary)

In [ ]:
# 分類にも使える: 選択肢を Literal で限定すると、それ以外の値は出力されない
class Sentiment(BaseModel):
    label: Literal["ポジティブ", "ネガティブ", "中立"]
    reason: str

for review in ["画面がきれいで操作も軽快。買ってよかった。", "届いてすぐ壊れた。サポートの対応も遅い。"]:
    r = client.models.generate_content(model=MODEL, contents=f"次のレビューの感情を判定してください。\n\n{review}",
                                       config=types.GenerateContentConfig(response_mime_type="application/json", response_schema=Sentiment))
    print(r.parsed.label, "|", r.parsed.reason)

- `response_mime_type="application/json"` と `response_schema` を指定すると、スキーマに従う JSON だけが生成されるように制約される。`response.parsed` で Pydantic のオブジェクトとして受け取れる。
- 抽出した文字列が原文に実在するかは保証されない。`e.text in text` で必ず検証する。文字位置が必要なら、自分で `text.find()` する。
- スキーマで表せない制約（値の範囲など）は、Pydantic の検証で確認する。`response.parsed` が `None` のときは、`response.text` を見て原因を調べる。
- スキーマの項目の説明 (`Field(description=...)`) も、モデルへの指示として使われる。